# Source-image spine analysis
Run cells in order. Select the first dendritic branch near the soma, then trace outward. 1 Select region → 2 Trace shaft → 3 Analyze → 4 Run Save result cell. Lower Spine threshold to include dim signal while keeping Shaft reference fixed. Inspect positive/rejected views. Detailed Chinese ImageJ steps: `IMAGEJ_MANUAL_CN.md`.

In [ ]:
# Keep caches and optional dependencies inside the project.
from pathlib import Path
import os
import sys
import importlib.util
import subprocess
import tempfile

PROJECT_DIR = Path.cwd().resolve()  # Start Jupyter from this repository.
INSTALL_MISSING = False
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_DIR = PROJECT_DIR / ".notebook_runtime"
DEPENDENCY_DIR = PROJECT_DIR / ".notebook_deps"
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
for key, name in {
    "TEMP": "tmp", "TMP": "tmp", "PIP_CACHE_DIR": "pip-cache",
    "MPLCONFIGDIR": "matplotlib", "IPYTHONDIR": "ipython",
    "JUPYTER_CONFIG_DIR": "jupyter-config", "JUPYTER_DATA_DIR": "jupyter-data",
    "JUPYTER_RUNTIME_DIR": "jupyter-runtime", "XDG_CACHE_HOME": "cache",
}.items():
    folder = RUNTIME_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(folder)
tempfile.tempdir = str(RUNTIME_DIR / "tmp")
for folder in (DEPENDENCY_DIR, PROJECT_DIR / "vendor"):
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
required = {
    "numpy": "numpy>=2.1,<3", "scipy": "scipy>=1.13,<2",
    "matplotlib": "matplotlib>=3.9,<4", "skimage": "scikit-image>=0.24,<0.27",
    "tifffile": "tifffile>=2025.1.10", "czifile": "czifile==2026.8.16",
    "imagecodecs": "imagecodecs==2026.8.16", "ipywidgets": "ipywidgets>=8,<9",
    "ipympl": "ipympl>=0.9,<0.11",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing and INSTALL_MISSING:
    subprocess.check_call([
        sys.executable, "-B", "-m", "pip", "install", "--target", str(DEPENDENCY_DIR),
        "--index-url", "https://pypi.org/simple", "--no-warn-script-location", *missing,
    ], env=os.environ.copy())
    if str(DEPENDENCY_DIR) not in sys.path:
        sys.path.insert(0, str(DEPENDENCY_DIR))
    importlib.invalidate_caches()
elif missing:
    raise ImportError(f"Missing dependencies: {missing}. Set INSTALL_MISSING=True and rerun.")

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, Image
from io import BytesIO
import html
get_ipython().run_line_magic("matplotlib", "widget")
matplotlib.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "figure.dpi": 100})
print("Project:", PROJECT_DIR)

In [ ]:
# Read original CZI pixels and their physical calibration.
from __future__ import annotations
import csv
import hashlib
import json
import math
import platform
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from xml.etree import ElementTree as ET
from scipy import ndimage as ndi
from skimage.filters import threshold_otsu
import tifffile
VERSION = "1.7.0-notebook"
STRUCTURE = np.ones((3, 3), bool)


@dataclass
class ImageData:
    image: np.ndarray
    sx: float
    sy: float
    source: str
    selection: dict = field(default_factory=dict)
    metadata_xml: str = ""

    def __post_init__(self):
        if self.image.ndim != 2 or min(self.image.shape) < 2:
            raise ValueError("A single-channel 2D image is required.")
        if not np.isfinite([self.sx, self.sy]).all() or min(self.sx, self.sy) <= 0:
            raise ValueError("X/Y pixel sizes must be positive (um/pixel).")
        if not np.isfinite(self.image).all():
            raise ValueError("The image contains NaN or Inf.")

@dataclass
class Settings:
    length_um: float = 20.0
    start_offset_um: float = 2.0
    half_width_um: float = 6.0
    threshold: float = 1000.0  # Fluorescence-positive pixels, including dim spine necks.
    shaft_threshold: float = 0.0  # 0 follows threshold; a positive value fixes the shaft reference.
    gaussian_um: float = 0.0
    background_radius_um: float = 0.0
    shaft_halfwidth_um: float = 0.0                                          
    shaft_window_um: float = 2.0
    shaft_min_halfwidth_um: float = 0.15
    shaft_max_halfwidth_um: float = 2.5
    shaft_margin_um: float = 0.0
    spine_reach_um: float = 3.0
    min_area_um2: float = 0.03
    max_area_um2: float = 4.0                          

    def validate(self):
        vals = asdict(self)
        if not np.isfinite(list(vals.values())).all():
            raise ValueError("Parameters must be finite numbers.")
        if min(self.length_um, self.half_width_um, self.threshold,
               self.shaft_window_um, self.shaft_min_halfwidth_um,
               self.shaft_max_halfwidth_um, self.spine_reach_um) <= 0:
            raise ValueError("Length, ROI half-width, threshold, shaft width/window and reach must be positive.")
        nonnegative = (self.start_offset_um, self.gaussian_um,
                       self.background_radius_um, self.shaft_halfwidth_um,
                       self.min_area_um2, self.max_area_um2, self.shaft_threshold)
        if min(nonnegative) < 0:
            raise ValueError("Offset, smoothing, background radius, fixed half-width and area limits cannot be negative.")
        if self.shaft_min_halfwidth_um > self.shaft_max_halfwidth_um:
            raise ValueError("Minimum shaft half-width cannot exceed the maximum.")
        if self.shaft_max_halfwidth_um >= self.half_width_um:
            raise ValueError("ROI half-width must exceed maximum shaft half-width.")
        if self.shaft_halfwidth_um >= self.half_width_um:
            raise ValueError("Fixed shaft half-width must be smaller than ROI half-width.")
        if self.max_area_um2 and self.max_area_um2 < self.min_area_um2:
            raise ValueError("Maximum candidate area cannot be smaller than the minimum.")

def scaling_from_xml(xml: str) -> tuple[float, float]:
    """CZI Scaling/Distance/Value stores metres; display-unit fields are not used."""
    root = ET.fromstring(xml)
    for node in root.iter():
        node.tag = node.tag.rsplit("}", 1)[-1]
    values = {}
    for item in root.findall(".//Scaling/Items/Distance"):
        if item.get("Id") in ("X", "Y"):
            values[item.get("Id")] = float(item.findtext("Value", "nan")) * 1e6
    xy = (values.get("X", float("nan")), values.get("Y", float("nan")))
    if not np.isfinite(xy).all() or min(xy) <= 0:
        raise ValueError("CZI has no valid X/Y calibration; verify it in ZEN or Bio-Formats.")
    return xy

def inspect_czi(path) -> list[dict]:
    from czifile import CziFile
    with CziFile(path) as czi:
        result = []
        for key, view in czi.scenes.items():
            result.append({"scene": int(key), "sizes": dict(view.sizes),
                           "starts": dict(zip(view.dims, view.start)),
                           "mpp": view.mpp, "channels": list(view.channels)})
        return result

def load_czi(path, scene=0, channel=0, time=0, z_start=0, z_end=None,
             projection="max") -> ImageData:
    """Channel/time/Z arguments are zero-based positions within the chosen scene."""
    from czifile import CziFile
    if projection not in ("max", "single"):
        raise ValueError("Projection must be max or single.")
    with CziFile(path) as czi:
        view = czi.scenes[int(scene)]
        original_sizes = dict(view.sizes)
        starts = dict(zip(view.dims, view.start))
        choice = {}
        for axis, index in (("C", channel), ("T", time)):
            size = original_sizes.get(axis, 1)
            if not isinstance(index, int) or not 0 <= index < size:
                raise ValueError(f"{axis} index is outside 0..{size - 1}.")
            if axis in original_sizes:
                choice[axis] = starts[axis] + index
        nz = original_sizes.get("Z", 1)
        z_end = nz - 1 if z_end is None else z_end
        if not 0 <= z_start <= z_end < nz:
            raise ValueError(f"Z range is outside 0..{nz - 1}; the end index is inclusive.")
        if "Z" in original_sizes:
            a = starts["Z"] + z_start
            choice["Z"] = a if projection == "single" else slice(a, starts["Z"] + z_end + 1)
        view = view(**choice)
        for axis, size in view.sizes.items():
            if axis not in ("Z", "Y", "X") and size > 1:
                raise ValueError(f"Unselected {axis} dimension ({size}); RGB, illumination and phase are not merged automatically.")
        xml = czi.metadata()
        sx, sy = view.mpp or scaling_from_xml(xml)
        arr = view.asarray()
        dims = list(view.dims)
        for i in reversed(range(len(dims))):
            if dims[i] not in ("Z", "Y", "X"):
                arr = np.take(arr, 0, axis=i)
                dims.pop(i)
        if "Z" in dims:
            i = dims.index("Z")
            arr = np.max(arr, axis=i) if projection == "max" else np.take(arr, 0, axis=i)
            dims.pop(i)
        if set(dims) != {"Y", "X"}:
            raise ValueError(f"Unsupported spatial dimensions: {dims}")
        arr = np.transpose(arr, (dims.index("Y"), dims.index("X")))
        selection = {"scene": scene, "channel": channel, "time": time,
                     "z_start": z_start, "z_end_inclusive": z_end,
                     "projection": projection, "source_sizes": original_sizes,
                     "source_bbox_global_px": list(view.bbox),
                     "calibration_source": "CZI full-resolution view.mpp (µm/pixel)"}
        return ImageData(arr, sx, sy, str(Path(path).resolve()), selection, xml)

In [ ]:
# Measure source pixels; separate shaft reference from the spine threshold.
def preprocess(data: ImageData, p: Settings) -> np.ndarray:
    p.validate()
    a = data.image.astype(np.float32)
    if p.gaussian_um:
        a = ndi.gaussian_filter(a, (p.gaussian_um / data.sy, p.gaussian_um / data.sx),
                                mode="reflect")
    if p.background_radius_um:
        ry = math.ceil(p.background_radius_um / data.sy)
        rx = math.ceil(p.background_radius_um / data.sx)
                                                                                     
        background = ndi.grey_opening(a, size=(2 * ry + 1, 2 * rx + 1), mode="reflect")
        a = np.maximum(a - background, 0)
    return a

def trim_path(points, sx, sy, length, offset=0.0):
    a = np.asarray(points, float)
    if a.ndim != 2 or a.shape[1] != 2 or len(a) < 2 or not np.isfinite(a).all():
        raise ValueError("Select at least two valid centerline points along one shaft.")
    if min(sx, sy, length) <= 0 or offset < 0:
        raise ValueError("Pixel sizes and length must be positive; offset must be nonnegative.")
    physical = a * [sx, sy]
    keep = np.r_[True, np.linalg.norm(np.diff(physical, axis=0), axis=1) > 1e-9]
    a, physical = a[keep], physical[keep]
    cum = np.r_[0, np.cumsum(np.linalg.norm(np.diff(physical, axis=0), axis=1))]
    end = offset + length
    if len(a) < 2 or cum[-1] < end - 1e-8:
        raise ValueError(f"Centerline length is {cum[-1]:.3f} um; at least {end:.3f} um is required. Add distal points.")
    s = np.r_[offset, cum[(cum > offset + 1e-9) & (cum < end - 1e-9)], end]
    path = np.column_stack([np.interp(s, cum, physical[:, i]) for i in (0, 1)])
                                                                                   
    for i in range(len(path) - 1):
        for j in range(i + 2, len(path) - 1):
            u, v = path[i + 1] - path[i], path[j + 1] - path[j]
            cross = u[0] * v[1] - u[1] * v[0]
            if abs(cross) < 1e-10:
                continue
            d = path[j] - path[i]
            t = (d[0] * v[1] - d[1] * v[0]) / cross
            q = (d[0] * u[1] - d[1] * u[0]) / cross
            if 0 <= t <= 1 and 0 <= q <= 1:
                raise ValueError("The centerline crosses itself; trace one shaft in order.")
    return path / [sx, sy]

@dataclass
class Geometry:
    path: np.ndarray
    bbox: tuple[int, int, int, int]
    s: np.ndarray
    t: np.ndarray
    corridor: np.ndarray
    grid_x: np.ndarray
    grid_y: np.ndarray
    s_axis: np.ndarray
    t_axis: np.ndarray
    length: float
    clipped: bool
    selection_box: tuple[int, int, int, int] | None = None

    def straighten(self, image, order=1, crop=False):
        x0, y0, _, _ = self.bbox
        return ndi.map_coordinates(image, [self.grid_y - (y0 if crop else 0),
                                           self.grid_x - (x0 if crop else 0)],
                                   order=order, mode="constant", cval=0,
                                   prefilter=False)

def geometry(data: ImageData, points, p: Settings, selection_box=None) -> Geometry:
    p.validate()
    path = trim_path(points, data.sx, data.sy, p.length_um, p.start_offset_um)
    h, w = data.image.shape
    if (path < 0).any() or (path[:, 0] > w - 1).any() or (path[:, 1] > h - 1).any():
        raise ValueError("The centerline must stay inside the image.")
    phys = path * [data.sx, data.sy]
    v = np.diff(phys, axis=0)
    lengths = np.linalg.norm(v, axis=1)
    cum = np.r_[0, np.cumsum(lengths)]
    x0, y0 = np.floor(path.min(axis=0) - p.half_width_um / np.array([data.sx, data.sy]) - 1).astype(int)
    x1, y1 = np.ceil(path.max(axis=0) + p.half_width_um / np.array([data.sx, data.sy]) + 2).astype(int)
    clipped = x0 < 0 or y0 < 0 or x1 > w or y1 > h
    x0, y0, x1, y1 = max(0, x0), max(0, y0), min(w, x1), min(h, y1)
    yy, xx = np.mgrid[y0:y1, x0:x1]
    xx, yy = xx * data.sx, yy * data.sy
    best = np.full(xx.shape, np.inf)
    longitudinal = np.zeros(xx.shape)
    transverse = np.zeros(xx.shape)
    endpoint_ok = np.zeros(xx.shape, bool)
    for i, (start, delta, seglen) in enumerate(zip(phys[:-1], v, lengths)):
        dx, dy = xx - start[0], yy - start[1]
        u = (dx * delta[0] + dy * delta[1]) / seglen ** 2
        clipped_u = np.clip(u, 0, 1)
        ex, ey = dx - clipped_u * delta[0], dy - clipped_u * delta[1]
        d2 = ex * ex + ey * ey
        use = d2 < best
        best[use] = d2[use]
        longitudinal[use] = cum[i] + clipped_u[use] * seglen
        sign = np.where(delta[0] * dy - delta[1] * dx < 0, -1, 1)
        transverse[use] = (np.sqrt(d2) * sign)[use]
        ok = ((u >= 0) if i == 0 else np.ones_like(u, bool))
        if i == len(lengths) - 1:
            ok &= u < 1                                                         
        endpoint_ok[use] = ok[use]
    corridor = endpoint_ok & (best <= p.half_width_um ** 2)
    if selection_box is not None:
        box = np.asarray(selection_box, float)
        if box.shape != (4,) or not np.isfinite(box).all():
            raise ValueError("Select a valid rectangle on the source image.")
        bx0, by0, bx1, by1 = box
        if not (0 <= bx0 < bx1 <= w and 0 <= by0 < by1 <= h):
            raise ValueError("The selected rectangle must stay inside the source image.")
        if np.any((path[:, 0] < bx0) | (path[:, 0] >= bx1) | (path[:, 1] < by0) | (path[:, 1] >= by1)):
            raise ValueError("The measured centerline must stay inside the selected rectangle.")
        corridor &= (xx / data.sx >= bx0) & (xx / data.sx < bx1) & (yy / data.sy >= by0) & (yy / data.sy < by1)
        selection_box = tuple(map(float, box))
    step = min(data.sx, data.sy)
    n = max(2, math.ceil(p.length_um / step))
    sa = (np.arange(n) + 0.5) * p.length_um / n
    nr = max(1, math.ceil(p.half_width_um / step))
    ta = np.linspace(-p.half_width_um, p.half_width_um, 2 * nr + 1)
    seg = np.minimum(np.searchsorted(cum, sa, side="right") - 1, len(v) - 1)
    frac = (sa - cum[seg]) / lengths[seg]
    centers = phys[seg] + frac[:, None] * v[seg]
    normals = np.column_stack([-v[seg, 1], v[seg, 0]]) / lengths[seg, None]
    coords = centers[None, :, :] + ta[:, None, None] * normals[None, :, :]
    return Geometry(path, tuple(map(int, (x0, y0, x1, y1))), longitudinal, transverse,
                    corridor, coords[:, :, 0] / data.sx, coords[:, :, 1] / data.sy,
                    sa, ta, float(cum[-1]), clipped, selection_box)

def _manual_edge(points, sa, default, half_width):
    if not points:
        return default
    a = np.asarray(points, float)
    if not np.isfinite(a).all() or (a[:, 0] < 0).any() or (a[:, 0] > sa[-1] + sa[0]).any():
        raise ValueError("Manual edge points exceed the centerline length.")
                                             
    unique = dict((float(s), abs(float(t))) for s, t in a)
    x = np.array(sorted(unique))
    y = np.array([unique[s] for s in x])
    if (y <= 0).any() or (y >= half_width).any():
        raise ValueError("Manual edges must be on the correct side and inside the ROI.")
    return np.interp(sa, x, y)

@dataclass
class Result:
    geometry: Geometry
    raw: np.ndarray
    corrected: np.ndarray
    positive: np.ndarray
    attached: np.ndarray
    shaft_domain: np.ndarray
    shaft: np.ndarray
    candidates: np.ndarray
    spines: np.ndarray
    excluded: np.ndarray
    upper: np.ndarray
    lower: np.ndarray
    components: list[dict]
    warnings: list[str]
    summary: dict
    automatic_spines: np.ndarray | None = None
    manual_added: np.ndarray | None = None
    manual_removed: np.ndarray | None = None
    brush_strokes: list = field(default_factory=list)

def brush_overrides(data, g, strokes):
    """Paint source-pixel centers using a calibrated round brush; later strokes win."""
    out = np.zeros(g.corridor.shape, np.int8)
    gx0, gy0, gx1, gy1 = g.bbox
    h, w = data.image.shape
    scale = np.array([data.sx, data.sy])
    for stroke in strokes:
        mode = stroke["mode"]
        radius = float(stroke["radius_um"])
        points = np.asarray(stroke["points_xy_source_px"], float)
        if mode not in ("add", "remove") or not np.isfinite(radius) or radius <= 0:
            raise ValueError("Brush mode must be add/remove and radius must be positive.")
        if points.ndim != 2 or points.shape[1] != 2 or len(points) == 0 or not np.isfinite(points).all():
            raise ValueError("Brush strokes require finite source X/Y points.")
        if (points < 0).any() or (points[:, 0] > w-1).any() or (points[:, 1] > h-1).any():
            raise ValueError("Brush points must stay inside the source image.")
        ends = points[1:] if len(points) > 1 else points
        for start, end in zip(points, ends):
            low = np.maximum([gx0, gy0], np.floor(np.minimum(start, end) - radius/scale)).astype(int)
            high = np.minimum([gx1, gy1], np.ceil(np.maximum(start, end) + radius/scale) + 1).astype(int)
            x0, y0 = low; x1, y1 = high
            if x0 >= x1 or y0 >= y1:
                continue
            yy, xx = np.ogrid[y0:y1, x0:x1]
            dx, dy = (xx-start[0])*data.sx, (yy-start[1])*data.sy
            vx, vy = (end-start)*scale
            length2 = vx*vx + vy*vy
            fraction = np.clip((dx*vx + dy*vy) / (length2 if length2 else 1), 0, 1)
            hit = (dx-fraction*vx)**2 + (dy-fraction*vy)**2 <= radius**2 + 1e-12
            patch = out[y0-gy0:y1-gy0, x0-gx0:x1-gx0]
            patch[hit] = 1 if mode == "add" else -1
    out[~g.corridor] = 0
    return out

def analyze(data: ImageData, corrected, g: Geometry, p: Settings,
            exclusions=None, upper_points=None, lower_points=None, brush_strokes=None) -> Result:
    p.validate()
    from matplotlib.path import Path as Polygon
    x0, y0, x1, y1 = g.bbox
    raw = data.image[y0:y1, x0:x1]
    a = corrected[y0:y1, x0:x1]
    exclusion = np.zeros(raw.shape, bool)
    yy, xx = np.mgrid[y0:y1, x0:x1]
    for vertices in exclusions or []:
        v = np.asarray(vertices, float)
        if v.ndim != 2 or v.shape[1] != 2 or len(v) < 3 or not np.isfinite(v).all():
            raise ValueError("An exclusion polygon requires at least three valid points.")
        exclusion |= Polygon(v).contains_points(np.column_stack([xx.ravel(), yy.ravel()]),
                                                 radius=1e-7).reshape(raw.shape)
    # Threshold on source pixels; no hole filling or intensity rescaling for measurement.
    positive = (a >= p.threshold) & g.corridor & ~exclusion
    display = g.straighten(corrected)
    # Estimate the shaft independently so lowering the spine threshold does not widen it.
    above = display >= (p.shaft_threshold or p.threshold)
    mid = len(g.t_axis) // 2
    dt = g.t_axis[1] - g.t_axis[0]
    warnings = []
    if p.shaft_halfwidth_um > 0:
        up = np.full(len(g.s_axis), p.shaft_halfwidth_um)
        down = up.copy()
    else:
                                                                                                                        
        up = np.full(len(g.s_axis), np.nan)
        down = up.copy()
        for j in range(len(g.s_axis)):
            if not above[mid, j]:
                continue
            u, d = mid, mid
            while u > 0 and above[u - 1, j]:
                u -= 1
            while d + 1 < len(g.t_axis) and above[d + 1, j]:
                d += 1
            up[j] = (mid - u + 0.5) * dt
            down[j] = (d - mid + 0.5) * dt
        valid = np.isfinite(up)
        if valid.sum() < 2:
            if not (upper_points and lower_points):
                raise ValueError("Insufficient positive centerline signal; lower the threshold, adjust the trace or set a fixed shaft half-width.")
            up[:] = down[:] = p.shaft_min_halfwidth_um
        else:
            for widths in (up, down):
                widths[:] = np.interp(g.s_axis, g.s_axis[valid], widths[valid])
                ds = g.length / len(g.s_axis)
                win = max(1, round(p.shaft_window_um / ds)) | 1
                widths[:] = ndi.median_filter(widths, size=win, mode="nearest")
        if valid.mean() < 0.9:
            warnings.append(f"Only {valid.mean():.0%} of centerline samples are positive; check threshold and trace.")
        if np.any(up > p.shaft_max_halfwidth_um) or np.any(down > p.shaft_max_halfwidth_um):
            warnings.append("Automatic shaft half-width reached its limit; inspect branches, thick shaft or merged spines.")
        up = np.clip(up, p.shaft_min_halfwidth_um, p.shaft_max_halfwidth_um)
        down = np.clip(down, p.shaft_min_halfwidth_um, p.shaft_max_halfwidth_um)
    up = np.clip(up + p.shaft_margin_um, p.shaft_min_halfwidth_um, p.half_width_um - dt / 2)
    down = np.clip(down + p.shaft_margin_um, p.shaft_min_halfwidth_um, p.half_width_um - dt / 2)
    up = _manual_edge(upper_points, g.s_axis, up, p.half_width_um)
    down = _manual_edge(lower_points, g.s_axis, down, p.half_width_um)
    mapped_up = np.interp(g.s, g.s_axis, up)
    mapped_down = np.interp(g.s, g.s_axis, down)
    band = (g.t >= -mapped_up) & (g.t <= mapped_down) & g.corridor & ~exclusion
    labels, _ = ndi.label(positive, STRUCTURE)
    # Only positive components connected to the shaft can become spine candidates.
    seeds = np.unique(labels[band & positive])
    seeds = seeds[seeds > 0]
    attached = np.isin(labels, seeds) & positive
    shaft = attached & band
    reach = (g.t >= -mapped_up - p.spine_reach_um) & (g.t <= mapped_down + p.spine_reach_um)
                                                                                                     
    candidate = attached & ~band
    lab, n = ndi.label(candidate, STRUCTURE)
    sizes = np.bincount(lab.ravel(), minlength=n + 1)
    keep = np.zeros(n + 1, bool)
    components = []
    for i, location in enumerate(ndi.find_objects(lab), 1):
        if location is None:
            continue
        loc = lab[location] == i
        area = float(sizes[i] * data.sx * data.sy)
        reason = "accepted"
        # Reject the entire component, rather than clipping a long branch into a spine.
        if np.any(loc & ~reach[location]):
            reason = "exceeds_reach"
        elif area < p.min_area_um2:
            reason = "below_min_area"
        elif p.max_area_um2 and area > p.max_area_um2:
            reason = "above_max_area"
        keep[i] = reason == "accepted"
        by, bx = np.nonzero(loc)
        components.append({"component_id": i, "area_um2": area,
                           "centroid_x_source_px": float(bx.mean() + location[1].start + x0),
                           "centroid_y_source_px": float(by.mean() + location[0].start + y0),
                           "status": reason})
    spine = keep[lab]
    automatic_spines = spine.copy() if brush_strokes else None
    manual_added = manual_removed = None
    if brush_strokes:
        edits = brush_overrides(data, g, brush_strokes)
        # Adding overrides automatic filters, but still requires positive signal outside the shaft.
        spine = (spine | ((edits == 1) & positive & ~band)) & (edits != -1)
        manual_added = spine & ~automatic_spines
        manual_removed = automatic_spines & ~spine
    rejected = sum(row["status"] != "accepted" for row in components)
    if rejected:
        warnings.append(f"Automatic filters rejected {rejected} protrusion components; see components.csv.")
    if g.clipped:
        warnings.append("The ROI is near an image edge; inspect truncated spines.")
    edge = g.corridor & ~ndi.binary_erosion(g.corridor, STRUCTURE)
    if np.any(spine & edge):
        warnings.append("Candidates touch the ROI boundary; outside area is excluded. Consider a wider ROI or another segment.")
    if np.any(positive & ~attached & ~spine):
        warnings.append("Detached positive signal was excluded; inspect dim necks and nearby fibers.")
    if np.issubdtype(data.image.dtype, np.integer):
        frac = float(np.mean(raw[g.corridor] == np.iinfo(data.image.dtype).max))
        if frac > 0.001:
            warnings.append(f"ROI: {frac:.1%} of pixels reach the storage maximum; inspect acquisition saturation.")
    area = float(spine.sum() * data.sx * data.sy)
    summary = {"metric": "2D fluorescence-positive protrusion area per centerline length",
               "spine_area_um2": area, "dendrite_length_um": g.length,
               "area_per_length_um2_per_um": area / g.length,
               "spine_pixels": int(spine.sum()), "pixel_area_um2": data.sx * data.sy,
               "shaft_positive_area_um2": float(shaft.sum() * data.sx * data.sy),
               "accepted_components_not_spine_count": int(ndi.label(spine, STRUCTURE)[1]) if brush_strokes else int(keep.sum()),
               "sx_um_per_pixel": data.sx, "sy_um_per_pixel": data.sy,
               "signal_threshold": p.threshold, "shaft_threshold": p.shaft_threshold or p.threshold}
    if brush_strokes:
        summary.update(automatic_spine_area_um2=float(automatic_spines.sum()*data.sx*data.sy),
                       manual_added_area_um2=float(manual_added.sum()*data.sx*data.sy),
                       manual_removed_area_um2=float(manual_removed.sum()*data.sx*data.sy),
                       manual_brush_strokes=len(brush_strokes))
    recorded_strokes = [{"mode": s["mode"], "radius_um": float(s["radius_um"]),
                         "points_xy_source_px": np.asarray(s["points_xy_source_px"], float).tolist()}
                        for s in brush_strokes or []]
    return Result(g, raw, a, positive, attached, band, shaft, candidate, spine,
                  exclusion, up, down, components, warnings, summary,
                  automatic_spines, manual_added, manual_removed, recorded_strokes)

def suggest_threshold(corrected, g=None):
    a = corrected if g is None else corrected[g.bbox[1]:g.bbox[3], g.bbox[0]:g.bbox[2]][g.corridor]
    if np.ptp(a) == 0:
        raise ValueError("Otsu requires nonconstant image/ROI intensities.")
    return max(float(threshold_otsu(a)), float(np.finfo(np.float32).eps))

In [ ]:
# Preserve source coordinates in annotations, masks and session records.
def write_tiff(path, array, sx, sy):
    tifffile.imwrite(path, np.asarray(array), imagej=True, resolution=(1 / sx, 1 / sy),
                     metadata={"axes": "YX", "unit": "um"})

def image_digest(image):
    return hashlib.sha256(np.ascontiguousarray(image).tobytes()).hexdigest()

def source_labels(data, result):
    labels = np.zeros(data.image.shape, np.uint8)
    x0, y0, x1, y1 = result.geometry.bbox
    crop = labels[y0:y1, x0:x1]
    crop[result.shaft] = 1
    crop[result.spines] = 2
    return labels

def source_annotation(data, result):
    from PIL import Image, ImageDraw
    high = max(1., float(np.percentile(data.image, 99.7)))
    gray = np.clip(data.image.astype(np.float32) * (255 / high), 0, 255).astype(np.uint8)
    rgb = np.zeros((*gray.shape, 3), np.uint8)
    rgb[:, :, 0] = gray
    x0, y0, x1, y1 = result.geometry.bbox
    crop = rgb[y0:y1, x0:x1]
    for mask, color in ((result.shaft, (26, 255, 51)), (result.spines, (255, 0, 255))):
        edge = mask & ~ndi.binary_erosion(mask, structure=STRUCTURE, border_value=0)
        crop[edge] = color
    image = Image.fromarray(rgb)
    draw = ImageDraw.Draw(image)
    bx0, by0, bx1, by1 = result.geometry.selection_box or result.geometry.bbox
    draw.rectangle((bx0, by0, bx1 - 1, by1 - 1), outline=(255, 255, 0), width=1)
    components, _ = ndi.label(result.spines, STRUCTURE)
    for location in ndi.find_objects(components):
        if location is not None:
            rows, cols = location
            draw.rectangle((max(x0, x0 + cols.start - 2), max(y0, y0 + rows.start - 2),
                            min(x1 - 1, x0 + cols.stop + 1), min(y1 - 1, y0 + rows.stop + 1)),
                           outline=(255, 0, 255), width=1)
    path = [tuple(point) for point in result.geometry.path]
    draw.line(path, fill=(0, 255, 255), width=1)
    for x, y in (path[0], path[-1]):
        draw.ellipse((x - 2, y - 2, x + 2, y + 2), outline=(255, 255, 0), width=1)
    return np.asarray(image)

def save_result(folder, data, points, p, result, exclusions=None,
                upper_points=None, lower_points=None):
    from PIL import Image
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=False)
    g = result.geometry
    labels = source_labels(data, result)
    annotated = source_annotation(data, result)
    write_tiff(folder / "source_raw.tif", data.image, data.sx, data.sy)
    write_tiff(folder / "source_labels.tif", labels, data.sx, data.sy)
    write_tiff(folder / "source_shaft_mask.tif", (labels == 1).astype(np.uint8) * 255, data.sx, data.sy)
    write_tiff(folder / "source_spine_mask.tif", (labels == 2).astype(np.uint8) * 255, data.sx, data.sy)
    for name, mask in (("source_auto_spine_mask", result.automatic_spines if result.automatic_spines is not None else result.spines),
                       ("source_manual_added_mask", result.manual_added), ("source_manual_removed_mask", result.manual_removed)):
        full = np.zeros(data.image.shape, np.uint8)
        if mask is not None:
            x0, y0, x1, y1 = g.bbox
            full[y0:y1, x0:x1] = mask.astype(np.uint8) * 255
        write_tiff(folder / (name + ".tif"), full, data.sx, data.sy)
    tifffile.imwrite(folder / "source_annotated.tif", annotated, imagej=True, photometric="rgb",
                     resolution=(1 / data.sx, 1 / data.sy), metadata={"axes": "YXS", "unit": "um"})
    Image.fromarray(annotated).save(folder / "source_annotated.png")
    masks = {"03_corridor": g.corridor, "04_positive": result.positive,
             "05_attached": result.attached, "06_shaft_domain": result.shaft_domain,
             "07_shaft_positive": result.shaft, "08_spine_mask": result.spines,
             "09_other_positive": result.positive & ~result.shaft & ~result.spines,
             "10_excluded": result.excluded, "11_candidates_before_filters": result.candidates}
    write_tiff(folder / "01_raw_crop.tif", result.raw, data.sx, data.sy)
    write_tiff(folder / "02_corrected_crop.tif", result.corrected, data.sx, data.sy)
    for name, mask in masks.items():
        write_tiff(folder / (name + ".tif"), mask.astype(np.uint8) * 255, data.sx, data.sy)
    ds = g.length / len(g.s_axis)
    dt = g.t_axis[1] - g.t_axis[0]
    write_tiff(folder / "straightened_intensity_VIEW_ONLY.tif",
               g.straighten(data.image.astype(np.float32)), ds, dt)
    write_tiff(folder / "straightened_spines_VIEW_ONLY.tif",
               g.straighten(result.spines.astype(np.uint8) * 255, order=0, crop=True), ds, dt)
    session = {"program_version": VERSION, "created_utc": datetime.now(timezone.utc).isoformat(),
               "source": data.source, "source_selection": data.selection,
               "source_shape_yx": list(data.image.shape), "source_dtype": str(data.image.dtype),
               "projection_sha256": image_digest(data.image),
               "sx_um_per_pixel": data.sx, "sy_um_per_pixel": data.sy,
               "settings": asdict(p), "trace_xy_source_px": np.asarray(points).tolist(),
               "selected_centerline_xy_source_px": g.path.tolist(),
               "crop_bbox_source_px_xyxy": list(g.bbox),
               "selection_box_source_px_xyxy": g.selection_box,
               "exclusion_polygons_xy_source_px": exclusions or [],
               "upper_boundary_points_s_t_um": upper_points or [],
               "lower_boundary_points_s_t_um": lower_points or [],
               "brush_strokes": result.brush_strokes,
               "brush_rule": "last stroke wins; add requires threshold-positive signal inside corridor, outside shaft/exclusions; manual edits override automatic connectivity/area/reach filters",
               "components_csv_scope": "automatic candidates before brush edits; final_components.csv describes the final mask",
               "summary": result.summary, "warnings": result.warnings,
               "source_annotation": {"shape_yx": list(data.image.shape),
                                     "labels": {"0": "unlabeled", "1": "shaft positive", "2": "spine candidate"},
                                     "colors": {"shaft": "green", "spines": "magenta", "centerline": "cyan", "ends": "yellow"},
                                     "scope": "selected segment in the analyzed channel/projection; original pixel coordinates",
                                     "display": "red fluorescence with green/magenta contours; quantitative pixels remain in source_raw.tif"},
               "area_definition": "count original source pixels; inclusive threshold; 8-connectivity; no hole filling",
               "runtime": {"python": platform.python_version(), "numpy": np.__version__,
                           "scipy": __import__("scipy").__version__,
                           "skimage": __import__("skimage").__version__,
                           "czifile": getattr(sys.modules.get("czifile"), "__version__", "not loaded")}}
    (folder / "session.json").write_text(json.dumps(session, ensure_ascii=False, indent=2), encoding="utf-8")
    if data.metadata_xml:
        (folder / "czi_metadata.xml").write_text(data.metadata_xml, encoding="utf-8")
    with (folder / "results.csv").open("w", newline="", encoding="utf-8-sig") as f:
        fields = ["source", *result.summary]
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerow({"source": data.source, **result.summary})
    with (folder / "components.csv").open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["component_id", "area_um2", "centroid_x_source_px",
                                             "centroid_y_source_px", "status"])
        writer.writeheader()
        writer.writerows(result.components)
    final_labels, _ = ndi.label(result.spines, STRUCTURE)
    with (folder / "final_components.csv").open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["component_id", "area_um2", "centroid_x_source_px", "centroid_y_source_px"])
        for i, location in enumerate(ndi.find_objects(final_labels), 1):
            if location is not None:
                yy, xx = np.nonzero(final_labels[location] == i)
                writer.writerow([i, len(xx)*data.sx*data.sy, float(xx.mean()+location[1].start+g.bbox[0]),
                                 float(yy.mean()+location[0].start+g.bbox[1])])
    with (folder / "shaft_boundaries.csv").open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["s_um", "upper_t_um", "lower_t_um"])
        writer.writerows(zip(g.s_axis, -result.upper, result.lower))
    with (folder / "centerline.csv").open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["x_source_px", "y_source_px", "x_crop_px", "y_crop_px"])
        writer.writerows((x, y, x - g.bbox[0], y - g.bbox[1]) for x, y in g.path)
    np.savetxt(folder / "centerline_xy.txt", g.path, fmt="%.10f", delimiter="\t")
    fig = Figure(figsize=(12, 7), layout="constrained")
    FigureCanvasAgg(fig)
    a1, a2 = fig.subplots(2, 1)
    bright = np.percentile(result.raw, 99.8)
    a1.imshow(result.raw, cmap="gray", vmin=0, vmax=max(1, bright))
    rgba = overlay(result)
    a1.imshow(rgba)
    a1.plot(g.path[:, 0] - g.bbox[0], g.path[:, 1] - g.bbox[1], color="cyan", lw=1)
    a1.set_title("Source pixels: shaft green; protrusions magenta; other positive amber")
    plot_horizontal_view(a2, data, result)
    a2.set(xlabel="Centerline distance (um)", ylabel="Normal offset (um)",
           title="Horizontal view: green shaft / magenta spines\n"
                 f"Area {result.summary['spine_area_um2']:.4f} um2 / {g.length:.4f} um = "
                 f"{result.summary['area_per_length_um2_per_um']:.5f} um2/um; VIEW ONLY")
    fig.savefig(folder / "review_overlay.png", dpi=160)
    fig.clear()
    fig.set_size_inches(9, 5)
    horizontal_ax = fig.subplots()
    for filled, name in ((True, "horizontal_filled_VIEW_ONLY.png"),
                         (False, "horizontal_outlines_VIEW_ONLY.png")):
        horizontal_ax.clear()
        plot_horizontal_view(horizontal_ax, data, result, filled=filled)
        fig.savefig(folder / name, dpi=160)
    return folder

def plot_horizontal_view(ax, data, result, high=None, filled=True):
    g = result.geometry
    high = max(1., float(np.percentile(data.image, 99.7)) if high is None else high)
    rgb = np.zeros((*g.grid_x.shape, 3), np.float32)
    rgb[:, :, 0] = np.clip(g.straighten(data.image.astype(np.float32)) / high, 0, 1)
    extent = (0, g.length, g.t_axis[-1], g.t_axis[0])
    ax.imshow(rgb, extent=extent, aspect="equal", interpolation="nearest", origin="upper")
    rgba = np.zeros((*g.grid_x.shape, 4), np.float32)
    # Nearest-neighbor sampling preserves mask classes; areas still use source pixels.
    for mask, color in ((result.shaft, (.1, 1, .2, .7)), (result.spines, (1, 0, 1, .85))):
        straight = g.straighten(mask.astype(np.uint8), order=0, crop=True).astype(bool)
        if filled:
            rgba[straight] = color
        elif straight.any():
            # Outline the same accepted pixels without covering fluorescence inside them.
            rows, cols = straight.shape
            xs = (np.arange(-1, cols + 1) + .5) * g.length / cols
            ys = g.t_axis[0] + (np.arange(-1, rows + 1) + .5) * np.ptp(g.t_axis) / rows
            ax.contour(xs, ys, np.pad(straight, 1), levels=[.5], colors=[color[:3]], linewidths=.8)
    if filled:
        ax.imshow(rgba, extent=extent, aspect="equal", interpolation="nearest", origin="upper")
    ax.plot(g.s_axis, -result.upper, color="lime", lw=1)
    ax.plot(g.s_axis, result.lower, color="lime", lw=1)
    style = "filled" if filled else "outlines only"
    ax.set(title=f"Horizontal view: {style} / green shaft / magenta spines",
           xlabel="Centerline distance (um)", ylabel="Normal offset (um)",
           xlim=(0, g.length), ylim=(g.t_axis[-1], g.t_axis[0]))

def overlay(r: Result):
    out = np.zeros((*r.raw.shape, 4), np.float32)
    out[r.positive & ~r.shaft & ~r.spines] = [1, .65, 0, .55]
    out[r.shaft] = [.1, 1, .2, .5]
    out[r.spines] = [1, 0, 1, .8]
    out[r.excluded] = [.2, .6, 1, .4]
    return out

In [ ]:
# Rectangle/trace edits are lightweight; Analyze runs the segmentation.
class NotebookAnalyzer:
    """Select and inspect interactively; result files are written only by the save cell."""
    parameter_labels = [
        ("length_um", "Segment length (um)"), ("start_offset_um", "Start offset (um)"),
        ("threshold", "Spine fluorescence threshold"), ("shaft_threshold", "Shaft reference (0=linked)"),
        ("spine_reach_um", "Max. spine reach (um)"), ("min_area_um2", "Min. candidate area (um2)"),
        ("max_area_um2", "Max. candidate area (0=unlimited)"),
        ("half_width_um", "Corridor half-width (um)"), ("gaussian_um", "Gaussian sigma (0=off)"),
        ("background_radius_um", "Background radius (0=off)"),
        ("shaft_halfwidth_um", "Fixed shaft half-width (0=auto)"),
        ("shaft_window_um", "Shaft median window (um)"),
        ("shaft_min_halfwidth_um", "Min. shaft half-width (um)"),
        ("shaft_max_halfwidth_um", "Max. shaft half-width (um)"),
        ("shaft_margin_um", "Shaft margin (um; signed)"),
    ]

    def __init__(self, data, settings, points=(), selection_box=None):
        from matplotlib.collections import LineCollection
        from matplotlib.patches import Rectangle, Ellipse
        from matplotlib.widgets import RectangleSelector
        self.data, self.selection_box = data, selection_box
        self.points = [list(map(float, point)) for point in points]
        self.exclusions, self.polygon, self.upper_points, self.lower_points = [], [], [], []
        self.brush_strokes, self._brush_stroke = [], None
        self.result = self.last_export = None
        self._preprocessed = self._geometry = self._last_geometry_key = None
        self._changing = False
        self.output_folder = PROJECT_DIR / "notebook_results" / (Path(data.source).stem + "_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
        self.inputs = {}
        fields = []
        for key, label in self.parameter_labels:
            control = widgets.FloatText(value=float(getattr(settings, key)), continuous_update=False,
                                        layout=widgets.Layout(width="115px"))
            self.inputs[key] = control
            fields.append(widgets.HBox([widgets.Label(label, layout=widgets.Layout(width="235px")), control]))
        grid = lambda children: widgets.GridBox(children, layout=widgets.Layout(
            grid_template_columns="repeat(2, minmax(350px, 1fr))", grid_gap="3px 8px"))
        self.mode = widgets.Dropdown(options=[("Draw rectangle", "region"), ("Trace centerline", "trace"),
            ("Upper shaft edge", "upper"), ("Lower shaft edge", "lower"), ("Exclusion polygon", "exclude"),
            ("Paint add spines", "brush_add"), ("Paint erase spines", "brush_remove")],
            value="trace", description="Edit", layout=widgets.Layout(width="260px"))
        self.layer = widgets.Dropdown(options=[("Accepted candidates", "accepted"),
            ("All threshold-positive", "positive"), ("Before area/reach filters", "candidates"),
            ("Rejected / detached signal", "rejected")], value="accepted", description="View",
            layout=widgets.Layout(width="320px"))
        self.slider = widgets.FloatSlider(value=settings.threshold, min=float(np.finfo(np.float32).eps),
            max=max(float(data.image.max()), settings.threshold, 1), step=max(1, float(data.image.max())/2000),
            continuous_update=False, description="Spine threshold", readout_format=".0f",
            style={"description_width": "105px"}, layout=widgets.Layout(width="600px"))
        self.status, self.warning, self.export_note, self.diagnostics = [widgets.HTML() for _ in range(4)]
        self._high = max(1., float(np.percentile(data.image, 99.7)))
        # Cache the display LUT once; changing it never changes measured source intensities.
        self._rgb = np.zeros((*data.image.shape, 3), np.uint8)
        self._rgb[:, :, 0] = np.clip(data.image.astype(float)*255/self._high, 0, 255).astype(np.uint8)
        with plt.ioff():
            self.figure, self.source_ax = plt.subplots(figsize=(9, 9))
            self.detail_figure, (self.crop_ax, self.straight_ax, self.outline_ax) = plt.subplots(3, 1, figsize=(9, 9))
        self.figure.subplots_adjust(left=.07, right=.99, bottom=.07, top=.95)
        self.detail_figure.subplots_adjust(left=.08, right=.98, bottom=.10, top=.94, hspace=.45)
        self.source_ax.set(title="Original CZI: green shaft / magenta candidates",
                           xlabel="Source X (pixel)", ylabel="Source Y (pixel)")
        self.raw_artist = self.source_ax.imshow(self._rgb)
        self.mask_artist = self.source_ax.imshow(np.zeros((1, 1, 4)), extent=(0, 1, 1, 0), interpolation="nearest")
        self.mask_artist.set_visible(False)
        self.trace_artist, = self.source_ax.plot([], [], "o-", color="cyan", ms=2, lw=.8)
        self.region_artist = Rectangle((0, 0), 1, 1, fill=False, edgecolor="yellow", lw=1)
        self.source_ax.add_patch(self.region_artist)
        self.box_artist = LineCollection([], colors="magenta", linewidths=.8)
        self.exclusion_artist = LineCollection([], colors="#48a6ff", linewidths=1)
        self.source_ax.add_collection(self.box_artist)
        self.source_ax.add_collection(self.exclusion_artist)
        self.brush_cursor = Ellipse((0,0), 1, 1, fill=False, edgecolor="cyan", lw=1, visible=False)
        self.source_ax.add_patch(self.brush_cursor)
        self.brush_path_artist, = self.source_ax.plot([], [], color="cyan", lw=1, visible=False)
        h, w = data.image.shape
        self.source_ax.set(xlim=(-.5, w-.5), ylim=(h-.5, -.5))
        # Keep artists and callbacks alive; rectangle motion must not rebuild the figures.
        self.selector = RectangleSelector(self.source_ax, self.region_selected, button=[1],
            minspanx=3, minspany=3, spancoords="pixels", useblit=True, interactive=True,
            drag_from_anywhere=True, props={"edgecolor":"yellow", "fill":False})
        self.selector.set_active(False)
        self.selector.set_visible(False)
        self.figure.canvas.header_visible = False
        self.figure.canvas.toolbar_visible = True
        self.figure.canvas.mpl_connect("button_press_event", self.clicked)
        self.figure.canvas.mpl_connect("motion_notify_event", self.brush_motion)
        self.figure.canvas.mpl_connect("button_release_event", self.brush_release)
        self.figure.canvas.mpl_connect("figure_leave_event", self.brush_release)
        self.detail_figure.canvas.mpl_connect("button_press_event", self.clicked)
        self.details = widgets.Accordion(children=[self.detail_figure.canvas], selected_index=None)
        self.details.set_title(0, "Crop / horizontal view / manual shaft edges")
        def button(label, action, primary=False):
            item = widgets.Button(description=label, button_style="primary" if primary else "",
                                  layout=widgets.Layout(width="155px"))
            item.on_click(lambda event: self.invoke(action))
            return item
        steps = widgets.HBox([button("1 Select region", self.begin_region),
                              button("2 Trace shaft", self.start_trace), button("3 Analyze / refresh", self.update, True)])
        actions = widgets.HBox([button("Undo point", self.undo), button("Finish exclusion", self.finish_polygon),
                               button("Undo exclusion", self.undo_exclusion)])
        self.brush_size = widgets.FloatSlider(value=1., min=.1, max=10., step=.1, continuous_update=False,
            description="Brush diameter (um)", style={"description_width":"135px"},
            layout=widgets.Layout(width="350px"))
        brush_controls = widgets.HBox([self.brush_size, button("Undo brush", self.undo_brush),
                                       button("Clear brush edits", self.clear_brush)])
        brush_help = widgets.HTML("<b>Brush:</b> choose Paint add/erase spines in Edit, then drag on Original CZI. "
            "Release to update. Add uses positive pixels outside the shaft; lower the spine threshold for dim signal.")
        advanced = widgets.VBox([grid(fields[7:]), widgets.HBox([
            button("Suggest Otsu", self.otsu), button("Auto shaft edges", self.clear_edges),
            button("Clear trace", self.clear_trace)])])
        self.parameters = widgets.Accordion(children=[advanced], selected_index=None)
        self.parameters.set_title(0, "Advanced preprocessing / shaft settings")
        help_text = widgets.HTML("<b>1 Region → 2 Trace away from soma → 3 Analyze → 4 Run Save result cell</b><br>"
            "Missing in positive view: lower spine threshold. Orange rejected signal: inspect area/reach limits. "
            "Green over a spine: correct the shaft edges.")
        self.panel = widgets.VBox([help_text, steps, widgets.HBox([self.mode, self.layer]), self.slider,
            grid(fields[:7]), actions, brush_controls, brush_help, self.parameters, self.status, self.warning, self.figure.canvas,
            self.diagnostics, self.details, self.export_note])
        self.slider.observe(self.threshold_changed, names="value")
        for control in self.inputs.values():
            control.observe(self.parameters_changed, names="value")
        self.mode.observe(self.mode_changed, names="value")
        self.layer.observe(lambda change: self.draw(), names="value")
        self.details.observe(lambda change: self.draw_details() if change["new"] == 0 else None, names="selected_index")
        if len(self.points) >= 2:
            self.update()
        else:
            self.draw()
            self.message("Step 1: select a rectangle near a soma-proximal dendritic branch.")

    def show(self):
        display(self.panel)

    def message(self, text, error=False):
        self.status.value = f'<div style="color:{"#a32929" if error else "#26364a"};padding:4px">{html.escape(str(text))}</div>'

    def invoke(self, action):
        try:
            action()
        except Exception as exc:
            self.message(exc, True)

    def settings(self):
        p = Settings(**{key: control.value for key, control in self.inputs.items()})
        p.validate()
        return p

    def corrected(self, p):
        key = (p.gaussian_um, p.background_radius_um)
        if self._preprocessed is None or self._preprocessed[0] != key:
            self._preprocessed = (key, preprocess(self.data, p))
        return self._preprocessed[1]

    def selected_geometry(self, p):
        key = (tuple(map(tuple, self.points)), p.length_um, p.start_offset_um, p.half_width_um,
               tuple(self.selection_box) if self.selection_box else None)
        if self._geometry is None or self._geometry[0] != key:
            if self._last_geometry_key is not None and self._last_geometry_key != key:
                self.upper_points, self.lower_points = [], []
            self._geometry = (key, geometry(self.data, self.points, p, self.selection_box))
            self._last_geometry_key = key
        return self._geometry[1]

    def update(self):
        if self._brush_stroke is not None:
            self.message("Release the mouse to finish the brush stroke before analyzing or saving.", True)
            return False
        try:
            p = self.settings()
            g = self.selected_geometry(p)
            self.result = analyze(self.data, self.corrected(p), g, p, self.exclusions, self.upper_points,
                                  self.lower_points, self.brush_strokes)
            s = self.result.summary
            self.message(f"Area {s['spine_area_um2']:.4f} um2 / {g.length:.4f} um = {s['area_per_length_um2_per_um']:.5f} um2/um")
            self.warning.value = html.escape("; ".join(self.result.warnings))
            # Show where positive area went, instead of silently hiding rejected components.
            by = {reason: sum(c["area_um2"] for c in self.result.components if c["status"] == reason)
                  for reason in ("below_min_area", "above_max_area", "exceeds_reach")}
            rows = [("Auto before filters", float(self.result.candidates.sum()*self.data.sx*self.data.sy)),
                    ("Auto accepted", s.get("automatic_spine_area_um2", s["spine_area_um2"])),
                    ("Brush added", s.get("manual_added_area_um2", 0.)),
                    ("Brush removed", s.get("manual_removed_area_um2", 0.)),
                    ("Final spines", s["spine_area_um2"]), ("Below min. area", by["below_min_area"]),
                    ("Above max. area", by["above_max_area"]), ("Beyond max. reach", by["exceeds_reach"])]
            self.diagnostics.value = "<b>Protrusion area (um2)</b><table>" + "".join(
                f"<tr><td>{label}</td><td style='padding-left:20px'>{value:.4f}</td></tr>" for label,value in rows) + "</table>"
            self.export_note.value = "Unsaved changes. Run the final Save result cell."
            self.draw()
            return True
        except Exception as exc:
            self.result = None
            self.message(exc, True)
            self.draw()
            return False

    def threshold_changed(self, change):
        if self._changing:
            return
        self._changing = True
        self.inputs["threshold"].value = change["new"]
        self._changing = False
        if self.result is not None:
            self.update()

    def parameters_changed(self, change):
        if self._changing:
            return
        self._changing = True
        if change["owner"] is self.inputs["threshold"]:
            self.slider.value = change["new"]
        self._changing = False
        if self.result is not None:
            self.update()
        else:
            self.message("Parameters changed. Finish tracing, then click 3 Analyze / refresh.")

    def mode_changed(self, change):
        if self._brush_stroke is not None:
            self.brush_release()
        self.brush_cursor.set_visible(False)
        if change["new"] in ("region", "trace", "brush_add", "brush_remove"):
            self.disable_navigation()
        region = change["new"] == "region"
        box = self.selection_box or (self.result.geometry.bbox if self.result is not None else None)
        if region and box:
            x0, y0, x1, y1 = box
            self.selector.extents = (x0, x1-1, y0, y1-1)
        self.selector.set_active(region)
        self.selector.set_visible(region)
        if change["new"] in ("upper", "lower"):
            self.details.selected_index = 0
        if change["new"].startswith("brush_"):
            self.layer.value = "accepted"
            self.message("Drag on Original CZI; release to apply. Undo brush removes the last stroke.")
        self.draw()

    def disable_navigation(self):
        toolbar = self.figure.canvas.toolbar
        mode = str(getattr(toolbar, "mode", "")).lower()
        if "pan" in mode: toolbar.pan()
        elif "zoom" in mode: toolbar.zoom()

    def begin_region(self):
        self.disable_navigation()
        if self.mode.value == "region": self.mode_changed({"new":"region"})
        else: self.mode.value = "region"

    def region_selected(self, start, end):
        if getattr(self.figure.canvas.toolbar, "mode", "") or any(v is None for v in [start.xdata,start.ydata,end.xdata,end.ydata]):
            return
        h,w = self.data.image.shape
        x0,x1 = sorted([start.xdata,end.xdata]); y0,y1 = sorted([start.ydata,end.ydata])
        self.selection_box = [max(0,int(np.floor(x0))),max(0,int(np.floor(y0))),min(w,int(np.ceil(x1))+1),min(h,int(np.ceil(y1))+1)]
        self.exclusions = []
        self.clear_trace()
        self.message("Step 1: move/resize the rectangle as needed. Click 2 Trace shaft when ready.")

    def start_trace(self):
        self.disable_navigation()
        self.clear_trace()
        self.mode.value = "trace"

    def clicked(self, event):
        canvas = getattr(event, "canvas", self.figure.canvas)
        if event.inaxes is None or getattr(canvas.toolbar, "mode", ""):
            return
        if event.button == 3:
            self.invoke(self.undo); return
        if event.button != 1:
            return
        point = [float(event.xdata),float(event.ydata)]
        mode = self.mode.value
        if mode.startswith("brush_"):
            if event.inaxes is not self.source_ax or self.result is None:
                self.message("Analyze a segment first, then paint on Original CZI.", True)
                return
            h,w = self.data.image.shape
            if not (0 <= point[0] <= w-1 and 0 <= point[1] <= h-1): return
            self._brush_stroke = {"mode":mode.removeprefix("brush_"), "radius_um":self.brush_size.value/2,
                                  "points_xy_source_px":[point]}
            self.brush_motion(event)
            return
        if mode in ("trace", "exclude") and event.inaxes in (self.source_ax,self.crop_ax):
            if event.inaxes is self.crop_ax:
                if self.result is None: return
                point = (np.array(point)+self.result.geometry.bbox[:2]).tolist()
            h,w = self.data.image.shape
            if not (0 <= point[0] < w and 0 <= point[1] < h): return
            if mode == "trace":
                if self.selection_box:
                    x0,y0,x1,y1 = self.selection_box
                    if not (x0 <= point[0] < x1 and y0 <= point[1] < y1):
                        self.message("Click inside the selected rectangle.", True); return
                self.points.append(point)
                self.result = None
                self.upper_points,self.lower_points = [],[]
                self.message(f"Step 2: {len(self.points)} points. Cover offset + length, then click 3 Analyze.")
            else:
                self.polygon.append(point)
                self.message(f"Exclusion: {len(self.polygon)} vertices. Click Finish exclusion after at least 3.")
            # Point placement updates only lightweight artists, not the segmentation.
            self.draw()
        elif mode in ("upper","lower") and event.inaxes in (self.straight_ax,self.outline_ax) and self.result is not None:
            if (mode == "upper" and point[1] >= 0) or (mode == "lower" and point[1] <= 0):
                self.message("Upper edge is negative; lower edge is positive.", True); return
            (self.upper_points if mode == "upper" else self.lower_points).append(point)
            self.update()

    def undo(self):
        mode = self.mode.value
        if mode.startswith("brush_"):
            self.undo_brush(); return
        if mode == "region":
            self.selection_box = None; self.clear_trace(); return
        target = {"trace":self.points,"exclude":self.polygon,"upper":self.upper_points,"lower":self.lower_points}[mode]
        if target: target.pop()
        if mode == "trace":
            self.result = None
            self.upper_points,self.lower_points = [],[]
            self.message("Trace changed. Click 3 Analyze when ready.")
            self.draw()
        elif mode == "exclude": self.draw()
        else: self.update()

    def clear_trace(self):
        self.points,self.polygon,self.upper_points,self.lower_points = [],[],[],[]
        self.brush_strokes, self._brush_stroke = [], None
        self.brush_cursor.set_visible(False)
        self.brush_path_artist.set_visible(False)
        self.result = self._geometry = self._last_geometry_key = None
        self.diagnostics.value = "Step 3 will show threshold/filter diagnostics after analysis."
        self.message("Step 2: trace outward from the soma-near branch, then click 3 Analyze.")
        self.draw()

    def brush_motion(self, event):
        if not self.mode.value.startswith("brush_") or self.result is None: return
        if event.inaxes is not self.source_ax or getattr(self.figure.canvas.toolbar, "mode", ""):
            self.brush_cursor.set_visible(False)
            if self._brush_stroke is not None: self.brush_release()
            self.figure.canvas.draw_idle()
            return
        point = [float(event.xdata),float(event.ydata)]
        h,w = self.data.image.shape
        if not (0 <= point[0] <= w-1 and 0 <= point[1] <= h-1): return
        stroke = self._brush_stroke
        radius = stroke["radius_um"] if stroke is not None else self.brush_size.value/2
        color = "magenta" if self.mode.value == "brush_add" else "cyan"
        self.brush_cursor.center = point
        self.brush_cursor.width, self.brush_cursor.height = 2*radius/self.data.sx, 2*radius/self.data.sy
        self.brush_cursor.set_edgecolor(color)
        self.brush_cursor.set_visible(True)
        if stroke is not None:
            points = stroke["points_xy_source_px"]
            if np.linalg.norm((np.array(point)-points[-1])*[self.data.sx,self.data.sy]) > .25*min(self.data.sx,self.data.sy):
                points.append(point)
            self.brush_path_artist.set_data(*np.asarray(points).T)
            self.brush_path_artist.set_color(color)
            self.brush_path_artist.set_visible(True)
        # Only the cursor/path changes while dragging; segmentation runs once on release.
        self.figure.canvas.draw_idle()

    def brush_release(self, event=None):
        if self._brush_stroke is None:
            if self.brush_cursor.get_visible():
                self.brush_cursor.set_visible(False)
                self.figure.canvas.draw_idle()
            return
        if event is not None and getattr(event, "inaxes", None) is self.source_ax:
            self.brush_motion(event)
        if self._brush_stroke is None: return
        self.brush_strokes.append(self._brush_stroke)
        self._brush_stroke = None
        self.brush_path_artist.set_visible(False)
        self.brush_cursor.set_visible(False)
        self.update()

    def undo_brush(self):
        if self._brush_stroke is not None:
            self._brush_stroke = None
        elif self.brush_strokes:
            self.brush_strokes.pop()
        self.brush_path_artist.set_visible(False)
        self.brush_cursor.set_visible(False)
        if self.result is not None: self.update()

    def clear_brush(self):
        self.brush_strokes, self._brush_stroke = [], None
        self.brush_path_artist.set_visible(False)
        self.brush_cursor.set_visible(False)
        if self.result is not None: self.update()

    def finish_polygon(self):
        if len(self.polygon) < 3: raise ValueError("Select at least 3 exclusion vertices.")
        self.exclusions.append(self.polygon.copy()); self.polygon = []
        self.update()

    def undo_exclusion(self):
        if self.exclusions: self.exclusions.pop()
        self.update()

    def clear_edges(self):
        self.upper_points,self.lower_points = [],[]
        self._changing = True
        self.inputs["shaft_halfwidth_um"].value = 0
        self._changing = False
        self.update()

    def otsu(self):
        p = self.settings()
        g = self.selected_geometry(p) if len(self.points) >= 2 else None
        value = suggest_threshold(self.corrected(p),g)
        self._changing = True
        self.inputs["threshold"].value = self.slider.value = value
        self._changing = False
        if self.result is not None: self.update()

    def export(self):
        if self._brush_stroke is not None: raise ValueError("Release the mouse before saving brush edits.")
        if self.polygon: raise ValueError("Finish or undo the pending exclusion polygon.")
        if self.result is None: raise ValueError("Click 3 Analyze before saving.")
        folder = self.output_folder.resolve()
        if not folder.is_relative_to(PROJECT_DIR):
            raise ValueError("Output must stay inside the project directory.")
        if folder.exists() and self.last_export != folder: raise FileExistsError("Output folder already exists.")
        # Stage new files inside the project before replacing the previous saved result from this run.
        with tempfile.TemporaryDirectory(prefix="annotation-",dir=RUNTIME_DIR/"tmp") as tmp:
            staging = save_result(Path(tmp)/"result",self.data,self.points,self.settings(),self.result,
                                  self.exclusions,self.upper_points,self.lower_points)
            folder.mkdir(parents=True,exist_ok=True)
            for path in staging.iterdir(): os.replace(path,folder/path.name)
        self.last_export = folder
        self.export_note.value = "Saved: <code>"+html.escape(str(folder/"source_annotated.png"))+"</code>"

    def draw(self, preserve=True):
        if self.points:
            xy = np.asarray(self.points); self.trace_artist.set_data(xy[:,0],xy[:,1])
        else: self.trace_artist.set_data([],[])
        polys = self.exclusions + ([self.polygon] if self.polygon else [])
        self.exclusion_artist.set_segments([np.asarray(poly+[poly[0]]) for poly in polys])
        box = self.selection_box or (self.result.geometry.bbox if self.result is not None else None)
        self.region_artist.set_visible(box is not None and self.mode.value != "region")
        if box:
            x0,y0,x1,y1 = box; self.region_artist.set_bounds(x0,y0,x1-x0,y1-y0)
        self.mask_artist.set_visible(self.result is not None)
        segments = []
        if self.result is not None:
            r = self.result; x0,y0,x1,y1 = r.geometry.bbox
            rgba = np.zeros((*r.raw.shape,4),np.float32)
            view = self.layer.value
            if view == "positive": rgba[r.positive] = (1,.75,0,.5)
            if view == "candidates": rgba[r.candidates] = (1,.65,0,.6)
            if view == "rejected":
                rgba[r.candidates & ~r.spines] = (1,.65,0,.7)
                rgba[r.positive & ~r.attached] = (0,1,1,.65)
            for mask,color in [(r.shaft,(.1,1,.2,1)),(r.spines,(1,0,1,1))]:
                if view != "accepted" and mask is r.spines: continue
                rgba[mask & ~ndi.binary_erosion(mask,STRUCTURE)] = color
            self.mask_artist.set_data(rgba)
            self.mask_artist.set_extent((x0-.5,x1-.5,y1-.5,y0-.5))
            if view == "accepted":
                for obj in ndi.find_objects(ndi.label(r.spines,STRUCTURE)[0]):
                    if obj is not None:
                        rows,cols=obj; bx0=x0+cols.start-2; bx1=x0+cols.stop+1; by0=y0+rows.start-2; by1=y0+rows.stop+1
                        segments.append([[bx0,by0],[bx1,by0],[bx1,by1],[bx0,by1],[bx0,by0]])
        self.box_artist.set_segments(segments)
        self.figure.canvas.draw_idle()
        if self.details.selected_index == 0: self.draw_details()

    def draw_details(self):
        # Hidden detail plots are rendered only when opened, never during rectangle motion.
        if self.result is None: return
        r=self.result; g=r.geometry
        self.crop_ax.clear(); self.straight_ax.clear(); self.outline_ax.clear()
        self.crop_ax.imshow(r.raw,cmap="gray",vmin=0,vmax=self._high)
        self.crop_ax.imshow(overlay(r))
        self.crop_ax.set(title="Crop: accepted magenta / other positive amber",xlabel="Crop X",ylabel="Crop Y")
        plot_horizontal_view(self.straight_ax,self.data,r,self._high)
        plot_horizontal_view(self.outline_ax,self.data,r,self._high,filled=False)
        self.detail_figure.canvas.draw_idle()

In [ ]:
# Edit input and initial parameters here. None chooses an Otsu starting value.
CZI_PATH = "path/to/your_image.czi"
SCENE = 0
CHANNEL = 0
TIME = 0
Z_START = 0
Z_END = None
PROJECTION = 'max'
LENGTH_UM = 15.0
START_OFFSET_UM = 2.0
SHAFT_MAX_HALFWIDTH_UM = 4.0
THRESHOLD = 14024.49000011921
SHAFT_THRESHOLD = 32744.2421875
SPINE_REACH_UM = 10.0
MIN_AREA_UM2 = 0.001
MAX_AREA_UM2 = 0.0
TRACE_POINTS = []
SELECTION_BOX = None

In [ ]:
# Load the original image; initialize a fixed shaft reference for spine-threshold tuning.

image_data = load_czi(CZI_PATH, scene=SCENE, channel=CHANNEL, time=TIME,
                      z_start=Z_START, z_end=Z_END, projection=PROJECTION)
settings = Settings(length_um=LENGTH_UM, start_offset_um=START_OFFSET_UM,
                    shaft_max_halfwidth_um=SHAFT_MAX_HALFWIDTH_UM,
                    shaft_threshold=SHAFT_THRESHOLD or 0, spine_reach_um=SPINE_REACH_UM,
                    min_area_um2=MIN_AREA_UM2, max_area_um2=MAX_AREA_UM2)
if THRESHOLD is None:
    initial_geometry = geometry(image_data, TRACE_POINTS, settings, SELECTION_BOX) if len(TRACE_POINTS) >= 2 else None
    settings.threshold = suggest_threshold(preprocess(image_data, settings), initial_geometry)
else:
    settings.threshold = float(THRESHOLD)
if SHAFT_THRESHOLD is None:
    settings.shaft_threshold = settings.threshold
if "analyzer" in globals():
    plt.close(analyzer.figure)
    plt.close(analyzer.detail_figure)
analyzer = NotebookAnalyzer(image_data, settings, TRACE_POINTS, SELECTION_BOX)
analyzer.show()
print(f"X/Y: {image_data.sx:.9g}/{image_data.sy:.9g} um/pixel")

### Save result
Run this cell after changing the selection or parameters.

In [ ]:
# Run this cell after reviewing the current selection and parameter changes.

if analyzer.polygon:
    raise ValueError("Finish or undo the pending exclusion polygon before saving.")
if not analyzer.update():
    raise ValueError("Select a sufficient centerline and valid parameters before saving.")
analyzer.export()
RESULT_DIR = analyzer.last_export
s = analyzer.result.summary
print(f"Candidate area: {s['spine_area_um2']:.6f} um2; length: {s['dendrite_length_um']:.6f} um")
print(f"Area / length: {s['area_per_length_um2_per_um']:.6f} um2/um")
print("Saved:", RESULT_DIR)
display(Image(filename=str(RESULT_DIR / "source_annotated.png")))